In [ ]:
import pandas as pd
import glob

def load_stock_data():
    files = glob.glob("./stock_data/*_historical_data.csv")

    price_df = pd.DataFrame()

    for file in files:
        ticker = file.split("/")[-1].split("_")[0]
        df = pd.read_csv(file)

        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)

        price_df[ticker] = df['Close']

    return price_df

prices = load_stock_data()
print(prices.head())

In [ ]:
returns = prices.pct_change().dropna()

expected_returns = returns.mean()
cov_matrix = returns.cov()

In [ ]:
pip install pulp

In [ ]:
import pulp

def optimize_portfolio(expected_returns, K=5):

    tickers = expected_returns.index.tolist()

    model = pulp.LpProblem("Portfolio_Optimization",
                           pulp.LpMaximize)

    # Decision variables
    x = pulp.LpVariable.dicts("weight", tickers, 0, 1)
    y = pulp.LpVariable.dicts("select", tickers, 0, 1, cat="Binary")

    # Objective: maximize return
    model += pulp.lpSum(expected_returns[i] * x[i]
                        for i in tickers)

    # Budget constraint
    model += pulp.lpSum(x[i] for i in tickers) == 1

    # Cardinality constraint
    model += pulp.lpSum(y[i] for i in tickers) <= K

    # Linking constraint
    for i in tickers:
        model += x[i] <= y[i]

    # Solve
    model.solve()

    solution = {i: x[i].value() for i in tickers}
    selected = {i: y[i].value() for i in tickers}

    return solution, selected

In [ ]:
weights, chosen = optimize_portfolio(expected_returns)

print("Optimal Weights:")
print(weights)

print("\nSelected Assets:")
print(chosen)

In [ ]:
import pulp
import numpy as np

def optimize_portfolio_MAD(returns, K=5, lam=50):

    tickers = returns.columns.tolist()
    T = len(returns)

    mean_returns = returns.mean()

    model = pulp.LpProblem("MAD_Portfolio", pulp.LpMaximize)

    # Decision variables
    x = pulp.LpVariable.dicts("weight", tickers, 0, 1)
    y = pulp.LpVariable.dicts("select", tickers, 0, 1, cat="Binary")

    # deviation variables
    z = pulp.LpVariable.dicts("deviation", range(T), 0)

    # ---------- OBJECTIVE ----------
    model += (
        pulp.lpSum(mean_returns[i]*x[i] for i in tickers)
        - lam*(1/T)*pulp.lpSum(z[t] for t in range(T))
    )

    # ---------- CONSTRAINTS ----------

    # Budget
    model += pulp.lpSum(x[i] for i in tickers) == 1

    # Cardinality
    model += pulp.lpSum(y[i] for i in tickers) == K

    # Linking constraints
    for i in tickers:
        model += x[i] <= y[i]
        model += x[i] >= 0.05*y[i]

    # MAD constraints
    for t in range(T):

        deviation = pulp.lpSum(
            (returns.iloc[t][i] - mean_returns[i]) * x[i]
            for i in tickers
        )

        model += z[t] >= deviation
        model += z[t] >= -deviation

    # Solve
    model.solve(pulp.PULP_CBC_CMD(msg=True))

    weights = {i: x[i].value() for i in tickers}
    selected = {i: y[i].value() for i in tickers}

    return weights, selected

In [ ]:
weights, selected = optimize_portfolio_MAD(returns)

print(weights)
print(selected)